In [3]:
import os
import re
import uuid
from typing import List, Dict

import fitz  # PyMuPDF
import pdfplumber
from tqdm import tqdm

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

/Users/pankajnapalchyal/Desktop/Data Science/RAG/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def detect_section_heading(text: str) -> str:
    """
    Detect possible section headings.

    Many rule books contain headings like:
    1. Introduction
    2.3 Compliance Rules
    Rule 4: Eligibility

    This function tries to detect them.
    """

    lines = text.split("\n")

    for line in lines[:5]:
        line = line.strip()

        if len(line) < 120 and re.match(r"^(\d+(\.\d+)*)\s+.+", line):
            return line

        if line.lower().startswith("rule"):
            return line

    return "unknown_section"


In [6]:
def extract_text(pdf_path: str) -> List[Document]:
    """
    Extract raw text from PDF with page metadata.
    """

    documents = []

    doc = fitz.open(pdf_path)

    for page_num in tqdm(range(len(doc)), desc="Extracting text"):
        page = doc.load_page(page_num)

        text = page.get_text()

        if not text.strip():
            continue

        section = detect_section_heading(text)

        documents.append(
            Document(
                page_content=text,
                metadata={
                    "page": page_num + 1,
                    "chunk_type": "text",
                    "section": section,
                    "source": "rule_book"
                }
            )
        )

    return documents

In [11]:
def extract_tables(pdf_path: str) -> List[Document]:

    table_docs = []

    with pdfplumber.open(pdf_path) as pdf:

        for page_num, page in enumerate(tqdm(pdf.pages, desc="Extracting tables")):

            tables = page.extract_tables()

            for table in tables:

                if not table:
                    continue

                header = [str(c) if c is not None else "" for c in table[0]]
                rows = [[str(c) if c is not None else "" for c in row] for row in table[1:]]

                md = "| " + " | ".join(header) + " |\n"
                md += "| " + " | ".join(["---"] * len(header)) + " |\n"

                for row in rows:
                    md += "| " + " | ".join(row) + " |\n"

                table_docs.append(
                    Document(
                        page_content=md,
                        metadata={
                            "page": page_num + 1,
                            "chunk_type": "table",
                            "section": "table_data",
                            "source": "rule_book"
                        }
                    )
                )

    return table_docs

In [9]:
PDF_PATH = "./pdf/tata_ipl_2024.pdf"
CHROMA_PATH = "./chroma_db"

# Chunking parameters tuned for rule documents
CHUNK_SIZE = 900
CHUNK_OVERLAP = 150

text_docs = extract_text(PDF_PATH)
print(f"\nText pages extracted: {len(text_docs)}")

Extracting text: 100%|██████████| 36/36 [00:00<00:00, 306.96it/s]


Text pages extracted: 36


In [28]:
table_docs = extract_tables(PDF_PATH)
print(f"Tables extracted: {len(table_docs)}")
# print sample table content
if table_docs:
    print("\nSample table content (Markdown format):")
    print(table_docs[0].page_content)

Extracting tables: 100%|██████████| 36/36 [00:04<00:00,  7.42it/s]

Tables extracted: 24

Sample table content (Markdown format):
|  | 2.1 |  |  | Excessive appealing during a Match |  |
| --- | --- | --- | --- | --- | --- |
| Note: |  |  | For the purpose of Article 2.1, ‘excessive’ may include (a) repeated appealing of the same decision;
(b) repeated appealing of different decisions when the bowler/fielder knows the batter is not out
with the intention of placing the Umpire under pressure; (c) charging or advancing towards the
Umpire in an aggressive manner when appealing; or (d) celebrating a dismissal without appealing
to the Umpire when a decision is required. It is not intended to prevent loud or enthusiastic
appealing. |  |  |
| Level 1 |  |  | ü |  |  |
| Level 2 |  |  | Not applicable |  |  |
| Level 3 |  |  | Not applicable |  |  |
| Level 4 |  |  | Not applicable |  |  |
|  |  |  |  |  |  |
| 2.2 |  |  |  | Abuse of cricket equipment or clothing, ground equipment or fixtures and fittings |  |
|  |  |  |  | during a Match. |  |
| Note: |  |  

In [13]:
def chunk_text_documents(docs: List[Document]) -> List[Document]:
    """
    Split long text into semantic chunks.

    Tables are not passed here (handled separately).
    """

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=[
            "\n\n",
            "\n",
            ". ",
            " "
        ]
    )

    chunks = []

    for doc in docs:

        splits = splitter.split_text(doc.page_content)

        for chunk in splits:

            chunks.append(
                Document(
                    page_content=chunk,
                    metadata=doc.metadata
                )
            )

    return chunks

In [14]:
# Chunk text
text_chunks = chunk_text_documents(text_docs)

print(f"\nText chunks created: {len(text_chunks)}")


Text chunks created: 135


In [15]:
# Combine
all_docs = text_chunks + table_docs

print(f"\nTotal chunks to embed: {len(all_docs)}")


Total chunks to embed: 159


In [16]:
def load_embedding_model():
    """
    Load high quality embedding model.

    Model: all-mpnet-base-v2
    Reason:
    - Strong semantic retrieval
    - Excellent for long documents
    - Open source
    """

    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-mpnet-base-v2"
    )

    return embeddings

In [20]:
# ----------------------------------------
# Store vectors in ChromaDB
# ----------------------------------------

def store_in_chroma(documents, embedding_model):

    os.makedirs(CHROMA_PATH, exist_ok=True)

    # Filter empty documents
    clean_docs = [
        doc for doc in documents
        if doc.page_content and doc.page_content.strip()
    ]

    if len(clean_docs) == 0:
        raise ValueError("No valid documents found after filtering empty chunks.")

    print(f"Storing {len(clean_docs)} valid chunks in ChromaDB")

    ids = [str(uuid.uuid4()) for _ in clean_docs]

    vectordb = Chroma.from_documents(
        documents=clean_docs,
        embedding=embedding_model,
        persist_directory=CHROMA_PATH,
        ids=ids
    )

    # vectordb.persist()

    return vectordb


In [18]:
# Load embedding model
embeddings = load_embedding_model()

/var/folders/8q/yv4mb1zs2j7ddg7j76tc8skh0000gn/T/ipykernel_73378/3921473616.py:12: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2063.73it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [21]:
# Store in Chroma
vectordb = store_in_chroma(all_docs, embeddings)
print("\nVector database created successfully.")

Storing 159 valid chunks in ChromaDB

Vector database created successfully.


In [33]:
def test_retrieval(vectordb):
    """
    Simple verification query.
    """

    query = "What are different levels of offenses and their range of permissible sanctions?"

    results = vectordb.similarity_search(query, k=5)

    print("\n=============================")
    print("RETRIEVAL TEST RESULTS")
    print("=============================\n")

    for i, doc in enumerate(results):

        print(f"Result {i+1}")
        print(f"Page:", doc.metadata["page"])
        print(f"Type:", doc.metadata["chunk_type"])
        print(f"Section:", doc.metadata["section"])
        print("-" * 50)
        print(doc.page_content[:1000])
        print("\n")


In [34]:
# Run retrieval test
test_retrieval(vectordb)


RETRIEVAL TEST RESULTS

Result 1
Page: 22
Type: table
Section: table_data
--------------------------------------------------
| RANGE OF |
| --- |
| PERMISSIBLE |
| SANCTIONS (ONE |
| OTHER PREVIOUS |
| OFFENCE) |



Result 2
Page: 22
Type: table
Section: table_data
--------------------------------------------------
| RANGE OF |
| --- |
| PERMISSIBLE |
| SANCTIONS (ONE |
| OTHER PREVIOUS |
| OFFENCE) |



Result 3
Page: 22
Type: table
Section: table_data
--------------------------------------------------
| RANGE OF |
| --- |
| PERMISSIBLE |
| SANCTIONS |
| (FIRST OFFENCE) |



Result 4
Page: 22
Type: table
Section: table_data
--------------------------------------------------
| RANGE OF |
| --- |
| PERMISSIBLE |
| SANCTIONS |
| (FIRST OFFENCE) |



Result 5
Page: 22
Type: table
Section: table_data
--------------------------------------------------
| RANGE OF |
| --- |
| PERMISSIBLE |
| SANCTIONS (THREE OR |
| MORE OTHER |
| PREVIOUS OFFENCES) |



